# Unit commitment – Gurobi model

Starter notebook for the UC model of the IFT 6575 project.

- Generator data: OR-Library instances parsed into `data/generators/out/` and loaded with `uc.data.load_generators`.
- Demand and spinning reserve: `*_load.txt` and `*_spinning_reserve.txt` (one value per hour).

**Licence note:** the installed Gurobi licence is *size-limited* (2000 variables and 2000 constraints).
The 10-generator instances fit; 20+ generators need the free academic licence.

In [ ]:
import sys
from pathlib import Path

import gurobipy as gp
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from gurobipy import GRB

# Make the repository importable whatever the notebook's working directory is.
ROOT = Path.cwd()
while not (ROOT / "uc").is_dir() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from uc.data import load_generators

## 1. Load an instance

In [ ]:
INSTANCE = "10_0_1_w"  # <n_generators>_0_<problem>_w
DATA_DIR = ROOT / "data" / "generators" / "out"

gen = load_generators(DATA_DIR / f"{INSTANCE}.csv")
demand = np.loadtxt(DATA_DIR / f"{INSTANCE}_load.txt")
reserve = np.loadtxt(DATA_DIR / f"{INSTANCE}_spinning_reserve.txt")

G = range(gen["n_g"])   # generators
T = range(len(demand))  # hours

print(f"{gen['n_g']} generators, {len(T)} hours")
print(f"demand: {demand.min():.0f}-{demand.max():.0f} MW, total capacity: {gen['max_out'].sum():.0f} MW")
pd.DataFrame({k: v for k, v in gen.items() if k != "n_g"})

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3))
ax.step(T, demand, where="post", label="demand")
ax.step(T, demand + reserve, where="post", linestyle="--", label="demand + reserve")
ax.set_xlabel("hour"); ax.set_ylabel("MW"); ax.legend()
plt.show()

## 2. Model

Variables, for each generator $g$ and hour $t$:
- $u_{gt} \in \{0,1\}$: on/off
- $v_{gt} \in \{0,1\}$: start-up, $w_{gt} \in \{0,1\}$: shut-down
- $p_{gt} \ge 0$: power output (MW)

Cost (linearised by the loader): no-load cost $\times u$ + marginal cost $\times p$ + start-up cost $\times v$.

In [ ]:
m = gp.Model("unit_commitment")

u = m.addVars(G, T, vtype=GRB.BINARY, name="u")
v = m.addVars(G, T, vtype=GRB.BINARY, name="v")
w = m.addVars(G, T, vtype=GRB.BINARY, name="w")
p = m.addVars(G, T, lb=0.0, name="p")

m.setObjective(
    gp.quicksum(gen["nol_cost"][g] * u[g, t] + gen["mar_cost"][g] * p[g, t] + gen["st_cost"][g] * v[g, t]
                for g in G for t in T),
    GRB.MINIMIZE,
)

### Core constraints

In [ ]:
# Demand balance
m.addConstrs((p.sum("*", t) == demand[t] for t in T), name="demand")

# Output limits when on, zero when off
m.addConstrs((p[g, t] >= gen["min_out"][g] * u[g, t] for g in G for t in T), name="min_out")
m.addConstrs((p[g, t] <= gen["max_out"][g] * u[g, t] for g in G for t in T), name="max_out")

# Start-up / shut-down logic (u at t = -1 is the initial condition)
def u_prev(g, t):
    return u[g, t - 1] if t > 0 else gen["initial_condition"][g]

m.addConstrs((u[g, t] - u_prev(g, t) == v[g, t] - w[g, t] for g in G for t in T), name="logic")

### To build next

Add each block in its own cell, re-solve, and compare the cost.

1. **Spinning reserve**: committed capacity covers demand + reserve for each hour.
2. **Ramping**: limit $p_{gt} - p_{g,t-1}$ using `op_rampup` / `st_rampup` and `op_rampdown` / `sh_rampdown`.
3. **Minimum up / down times**: `min_up`, `min_down` (e.g. the turn-on / turn-off inequalities on $v$ and $w$).
4. *(optional)* Notification time: `notification`.

In [ ]:
# TODO 1 – spinning reserve

In [ ]:
# TODO 2 – ramping

In [ ]:
# TODO 3 – minimum up / down times

## 3. Solve

In [ ]:
m.update()
print(f"{m.NumVars} variables, {m.NumConstrs} constraints (size-limited licence: 2000 each)")

m.Params.MIPGap = 1e-4
m.Params.TimeLimit = 60
m.optimize()

if m.SolCount:
    print(f"\ncost: {m.ObjVal:,.2f}   gap: {m.MIPGap:.2%}")

## 4. Results

In [ ]:
commit = pd.DataFrame([[round(u[g, t].X) for t in T] for g in G], index=[f"g{g}" for g in G], columns=list(T))
power = pd.DataFrame([[p[g, t].X for t in T] for g in G], index=commit.index, columns=list(T))

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 6), sharex=True)
ax1.stackplot(list(T), power.values, labels=power.index, step="post")
ax1.step(T, demand, where="post", color="black", label="demand")
ax1.set_ylabel("MW"); ax1.legend(ncol=6, fontsize=7)
ax2.imshow(commit.values, aspect="auto", cmap="Greens", interpolation="nearest")
ax2.set_yticks(range(len(commit.index)), commit.index)
ax2.set_xlabel("hour"); ax2.set_title("commitment (green = on)")
plt.tight_layout(); plt.show()